# Week 6: Unsupervised Learning & Clustering
## Notebook 12: OPTICS: Ordering Points to Identify the Clustering Structure

### 1. Conceptual Intuition & Theoretical Background
Like DBSCAN, **OPTICS (Ankerst et al., 1999)** is a density-based algorithm. However, while DBSCAN outputs a rigid flat partitioning based on a single $\varepsilon$, OPTICS creates an **ordered reachability-distance plot** that visually captures clusters of varying densities.

**Core Definitions:**
1. **Core Distance:** Minimum radius $\varepsilon'$ such that point $p$ contains at least $\text{MinPts}$ neighbors:
$$\text{core-dist}_{\varepsilon, \text{MinPts}}(p) = \begin{cases} \text{Distance to MinPts-th nearest neighbor} & \text{if } |N_\varepsilon(p)| \ge \text{MinPts} \\ \text{UNDEFINED} & \text{otherwise} \end{cases}$$

2. **Reachability Distance of $p$ from core object $o$:**
$$\text{reach-dist}(p, o) = \max\left( \text{core-dist}(o),\ \text{dist}(o, p) \right)$$

3. **Reachability Plot Interpretation:**
Dips (valleys) represent dense clusters; peaks represent noise or inter-cluster boundaries.

---

### 2. Scratch Python Implementation: OPTICS Ordering Engine

In [ ]:
import numpy as np

def optics_ordering_scratch(X, min_pts=4, max_eps=10.0):
    """
    Computes OPTICS cluster ordering and reachability distances.
    """
    n_samples = len(X)
    diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
    dist_mat = np.sqrt(np.sum(diff ** 2, axis=-1))
    
    # Calculate core distances
    core_distances = np.full(n_samples, np.inf)
    for i in range(n_samples):
        neighbors = np.where(dist_mat[i] <= max_eps)[0]
        if len(neighbors) >= min_pts:
            sorted_dists = np.sort(dist_mat[i, neighbors])
            core_distances[i] = sorted_dists[min_pts - 1]
            
    processed = np.zeros(n_samples, dtype=bool)
    ordered_list = []
    reachability = np.full(n_samples, np.inf)
    
    # Priority Queue tracking candidate seeds: list of (index, reachability_dist)
    for start_idx in range(n_samples):
        if processed[start_idx]:
            continue
            
        # Process starting point
        processed[start_idx] = True
        ordered_list.append(start_idx)
        
        if np.isneginf(core_distances[start_idx]) or np.isposinf(core_distances[start_idx]):
            continue
            
        # Initialize seeds priority list
        seeds = []
        for j in range(n_samples):
            if not processed[j] and dist_mat[start_idx, j] <= max_eps:
                new_reach = max(core_distances[start_idx], dist_mat[start_idx, j])
                reachability[j] = new_reach
                seeds.append((j, new_reach))
                
        seeds.sort(key=lambda x: x[1])
        
        while seeds:
            curr_idx, curr_reach = seeds.pop(0)
            if processed[curr_idx]:
                continue
                
            processed[curr_idx] = True
            ordered_list.append(curr_idx)
            
            if not np.isposinf(core_distances[curr_idx]):
                for k in range(n_samples):
                    if not processed[k] and dist_mat[curr_idx, k] <= max_eps:
                        new_reach = max(core_distances[curr_idx], dist_mat[curr_idx, k])
                        if new_reach < reachability[k]:
                            reachability[k] = new_reach
                            seeds.append((k, new_reach))
                            
                seeds.sort(key=lambda x: x[1])
                
    return ordered_list, reachability[ordered_list]

# Generate Multi-Density Clusters
np.random.seed(42)
c1 = np.random.normal(loc=[-4, 0], scale=0.3, size=(40, 2))  # Dense cluster
c2 = np.random.normal(loc=[4, 0], scale=1.2, size=(40, 2))   # Sparse cluster
X_data = np.vstack([c1, c2])

ordered_pts, reach_dists = optics_ordering_scratch(X_data, min_pts=4, max_eps=10.0)

print("--- OPTICS Reachability Diagnostics ---")
print(f"Total Ordered Points:        {len(ordered_pts)}")
print(f"Dense Cluster Min Reach:     {np.min(reach_dists[:40]):.3f}")
print(f"Sparse Cluster Mean Reach:   {np.mean(reach_dists[40:]):.3f}")
print("Deep reachability valley separates dense cluster from sparse cluster!")

### 3. Step-by-Step Code Walkthrough
1. **Core Distance Calculation:** Evaluates radius required to encapsulate $\text{MinPts}$ neighbors.
2. **Priority Seed Traversal:** Expands cluster boundaries by visiting points with the lowest reachability first.
3. **Reachability Valleys:** Valleys represent cluster cores; peaks indicate cluster boundaries.

---

### 4. Industry Applications & Trade-offs
- **Astronomical Star Cluster Mapping:** Identifying star clusters of varying densities against background galactic noise.
- **Trade-off:** Priority queue maintenance increases runtime relative to standard DBSCAN.

---

### 5. Top Interview Questions & Answers
**Q1: What is the main difference between DBSCAN and OPTICS?**
> **Answer:** DBSCAN outputs a single fixed clustering assignment based on a rigid $\varepsilon$. OPTICS outputs an ordered reachability graph that represents all possible clusterings across a continuum of density thresholds simultaneously.

**Q2: How do you extract flat clusters from an OPTICS reachability plot?**
> **Answer:** Either by drawing a horizontal cutoff line across the reachability plot (equivalent to DBSCAN with that $\varepsilon$), or using the **$\xi$ (Xi) steep-down/steep-up algorithm** to identify hierarchical cluster valleys.